# Smartphone Addiction Prediction — Optimized Pipeline v2
**Target Metric:** ROC-AUC ≥ 0.97  
**Models:** LightGBM (deep) · XGBoost (shallow) · CatBoost (medium) · RealMLP  
**Key Changes from v1:**
1. Feature count cut from 60+ → ~20 (removed monotonic transforms, redundant ratios, noisy indicators)
2. Native NaN handling — raw features keep their NaN for tree models to learn optimal missing splits
3. Differentiated hyperparameters per model for maximum ensemble diversity
4. RealMLP added as 4th model for neural-network-level diversity
5. Hill climbing blending replaces Nelder-Mead
6. Rank averaging for final submission


In [7]:
import os, warnings, time
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
!pip install -q pytabkit

warnings.filterwarnings('ignore')

# Try to import RealMLP — if not installed, we skip it gracefully
try:
    from pytabkit.models.sklearn.sklearn_interfaces import RealMLP_TD_Classifier
    HAS_REALMLP = True
    print("RealMLP available — will use as 4th ensemble member.")
except ImportError:
    HAS_REALMLP = False
    print("RealMLP not installed. Install with: pip install pytabkit")
    print("Proceeding with 3-model tree ensemble only.")

print("All imports loaded.")

RealMLP available — will use as 4th ensemble member.
All imports loaded.


In [8]:
# Auto-detect paths (works on both Kaggle and local)
if os.path.exists('/kaggle/input/competitions/playground-series-s6e8/train.csv'):
    train_path = '/kaggle/input/competitions/playground-series-s6e8/train.csv'
    test_path = '/kaggle/input/competitions/playground-series-s6e8/test.csv'
elif os.path.exists('data/train.csv'):
    train_path = 'data/train.csv'
    test_path = 'data/test.csv'
else:
    raise FileNotFoundError("Cannot find train.csv — check your paths.")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print(f"Train: {train.shape[0]:,} rows × {train.shape[1]} columns")
print(f"Test:  {test.shape[0]:,} rows × {test.shape[1]} columns")
print(f"\nTarget balance: {train['addicted_label'].value_counts(normalize=True).to_dict()}")
print(f"Missing rates (train):")
for col in train.columns:
    miss = train[col].isna().mean()
    if miss > 0:
        print(f"  {col}: {miss:.1%}")

Train: 691,369 rows × 14 columns
Test:  296,302 rows × 13 columns

Target balance: {1: 0.7094243450313797, 0: 0.29057565496862026}
Missing rates (train):
  age: 4.2%
  daily_screen_time_hours: 13.9%
  social_media_hours: 19.4%
  gaming_hours: 18.3%
  work_study_hours: 7.5%
  sleep_hours: 6.4%
  notifications_per_day: 9.8%
  app_opens_per_day: 11.7%
  weekend_screen_time: 16.2%
  gender: 4.2%
  stress_level: 8.0%
  academic_work_impact: 6.4%


In [9]:
def build_features(df):
    """
    Lean feature engineering — only features that add genuine signal for tree models.
    
    Design principles:
    - NO monotonic transforms (log, square, cube) — trees find the same splits
    - NO redundant ratio pairs — keep only the most informative one
    - NO individual _isna flags — keep only aggregate missing_count
    - NO numeric re-encodings of categoricals — trees handle categoricals natively
    - KEEP NaN in raw features — LGB/XGB/CatBoost handle NaN optimally
    - Only compute ratios/interactions with clear domain meaning
    """
    df = df.copy()
    
    # --- Raw feature references (keep NaN for tree models) ---
    screen  = df['daily_screen_time_hours']
    social  = df['social_media_hours']
    gaming  = df['gaming_hours']
    work    = df['work_study_hours']
    sleep   = df['sleep_hours']
    notifs  = df['notifications_per_day']
    opens   = df['app_opens_per_day']
    weekend = df['weekend_screen_time']
    age     = df['age']
    
    # --- For computed features, use median fill to avoid NaN propagation ---
    s = screen.fillna(screen.median())
    sl = sleep.fillna(sleep.median())
    n = notifs.fillna(notifs.median())
    o = opens.fillna(opens.median())
    w = work.fillna(work.median())
    so = social.fillna(social.median())
    g = gaming.fillna(gaming.median())
    wk = weekend.fillna(weekend.median())
    
    # 1. Aggregate missingness (single feature, not 11 flags)
    raw_cols = ['daily_screen_time_hours', 'social_media_hours', 'gaming_hours',
                'work_study_hours', 'sleep_hours', 'notifications_per_day',
                'app_opens_per_day', 'weekend_screen_time', 'age',
                'stress_level', 'academic_work_impact']
    df['missing_count'] = df[raw_cols].isna().sum(axis=1)
    
    # 2. Meaningful domain combinations
    df['recreational_hours'] = so + g                          # total leisure screen time
    df['unproductive_screen'] = s - w                          # screen not spent on work
    df['weekend_shift'] = wk - s                               # behavioral change on weekends
    
    # 3. Meaningful intensity ratios
    df['notif_per_screen_hour'] = n / (s + 0.1)               # notification bombardment intensity
    df['app_session_duration'] = (s * 60) / (o + 1)           # avg minutes per app open
    df['sleep_deficit_x_screen'] = np.maximum(0, 8 - sl) * s  # sleep debt amplified by screen
    
    # 4. Key behavioral interactions
    df['compulsive_index'] = np.log1p(s * (n + 1) * (o + 1))  # composite engagement
    df['screen_x_stress'] = s * df['stress_level'].map(
        {'Low': 1, 'Medium': 2, 'High': 3}).fillna(2)         # screen × stress
    df['screen_by_age'] = s / (age.fillna(age.median()) + 1)  # age-adjusted screen
    
    # 5. Keep categoricals as category dtype
    for col in ['gender', 'stress_level', 'academic_work_impact']:
        df[col] = df[col].astype('category')
    
    return df

# Apply
X = build_features(train.drop(columns=['id', 'addicted_label']))
y = train['addicted_label']
X_test = build_features(test.drop(columns=['id']))
X_test = X_test.reindex(columns=X.columns)

print(f"Feature count: {X.shape[1]} (down from ~60 in v1)")
print(f"Features: {list(X.columns)}")

Feature count: 22 (down from ~60 in v1)
Features: ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time', 'gender', 'stress_level', 'academic_work_impact', 'missing_count', 'recreational_hours', 'unproductive_screen', 'weekend_shift', 'notif_per_screen_hour', 'app_session_duration', 'sleep_deficit_x_screen', 'compulsive_index', 'screen_x_stress', 'screen_by_age']


In [10]:
cat_cols = ['gender', 'stress_level', 'academic_work_impact']

# XGBoost needs integer-encoded categoricals
X_xgb = X.copy()
X_test_xgb = X_test.copy()
for col in cat_cols:
    X_xgb[col] = X_xgb[col].cat.codes.replace(-1, np.nan)
    X_test_xgb[col] = X_test_xgb[col].cat.codes.replace(-1, np.nan)

# CatBoost needs string categoricals
X_cat = X.copy()
X_test_cat = X_test.copy()
for col in cat_cols:
    X_cat[col] = X_cat[col].astype(str)
    X_test_cat[col] = X_test_cat[col].astype(str)

# RealMLP needs fully numeric, no NaN
if HAS_REALMLP:
    X_mlp = X.copy()
    X_test_mlp = X_test.copy()
    for col in cat_cols:
        le = LabelEncoder()
        combined = pd.concat([X_mlp[col].astype(str), X_test_mlp[col].astype(str)])
        le.fit(combined)
        X_mlp[col] = le.transform(X_mlp[col].astype(str)).astype(float)
        X_test_mlp[col] = le.transform(X_test_mlp[col].astype(str)).astype(float)
    # Fill NaN with median for MLP
    for col in X_mlp.columns:
        median_val = X_mlp[col].median()
        X_mlp[col] = X_mlp[col].fillna(median_val)
        X_test_mlp[col] = X_test_mlp[col].fillna(median_val)

print("Data variants prepared.")

Data variants prepared.


In [11]:
def hill_climb_blend(oof_dict, y_true, n_rounds=200):
    """
    Greedy hill climbing for ensemble weight optimization.
    More robust than Nelder-Mead — iteratively adds the model
    that improves the blend the most.
    """
    names = list(oof_dict.keys())
    oofs = [oof_dict[n] for n in names]
    
    # Initialize with the best single model
    best_single_idx = np.argmax([roc_auc_score(y_true, o) for o in oofs])
    counts = [0] * len(oofs)
    counts[best_single_idx] = 1
    blend = oofs[best_single_idx].copy()
    best_auc = roc_auc_score(y_true, blend)
    
    for rnd in range(n_rounds):
        improved = False
        for i in range(len(oofs)):
            total = sum(counts) + 1
            trial = (blend * sum(counts) + oofs[i]) / total
            trial_auc = roc_auc_score(y_true, trial)
            if trial_auc > best_auc + 1e-7:
                best_auc = trial_auc
                best_idx = i
                improved = True
        
        if not improved:
            break
        
        counts[best_idx] += 1
        blend = sum(c * o for c, o in zip(counts, oofs)) / sum(counts)
    
    total = sum(counts)
    weights = {n: c / total for n, c in zip(names, counts)}
    return weights, best_auc

print("Hill climbing blender defined.")

Hill climbing blender defined.


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# OOF containers
oof = {'lgb': np.zeros(len(train)),
       'xgb': np.zeros(len(train)),
       'cat': np.zeros(len(train))}
test_preds = {'lgb': np.zeros(len(test)),
              'xgb': np.zeros(len(test)),
              'cat': np.zeros(len(test))}

if HAS_REALMLP:
    oof['mlp'] = np.zeros(len(train))
    test_preds['mlp'] = np.zeros(len(test))

fold_scores = {k: [] for k in oof.keys()}

print("=" * 65)
print("TRAINING 5-FOLD CROSS-VALIDATION")
print("=" * 65)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    print(f"\n{'='*65}")
    print(f"FOLD {fold + 1} / 5")
    print(f"{'='*65}")
    
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
    t0 = time.time()
    
    # ----------------------------------------------------------
    # 1. LightGBM — DEEP & EXPRESSIVE
    # ----------------------------------------------------------
    lgb_model = lgb.LGBMClassifier(
        objective='binary',
        metric='auc',
        boosting_type='gbdt',
        learning_rate=0.02,
        num_leaves=127,
        max_depth=10,
        min_child_samples=30,
        subsample=0.7,
        colsample_bytree=0.6,
        reg_alpha=0.05,
        reg_lambda=0.5,
        n_estimators=5000,
        random_state=42 + fold,
        verbose=-1
    )
    lgb_model.fit(
        X.iloc[tr_idx], y_tr,
        eval_set=[(X.iloc[va_idx], y_va)],
        callbacks=[lgb.early_stopping(200, verbose=False)]
    )
    oof['lgb'][va_idx] = lgb_model.predict_proba(X.iloc[va_idx])[:, 1]
    test_preds['lgb'] += lgb_model.predict_proba(X_test)[:, 1] / 5
    auc_lgb = roc_auc_score(y_va, oof['lgb'][va_idx])
    fold_scores['lgb'].append(auc_lgb)
    print(f"  LightGBM AUC: {auc_lgb:.5f}  (iters: {lgb_model.best_iteration_})")
    
    # ----------------------------------------------------------
    # 2. XGBoost — SHALLOW & REGULARIZED
    # ----------------------------------------------------------
    xgb_model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        tree_method='hist',
        learning_rate=0.05,
        max_depth=4,
        min_child_weight=80,
        subsample=0.9,
        colsample_bytree=0.8,
        reg_alpha=0.5,
        reg_lambda=2.0,
        n_estimators=5000,
        early_stopping_rounds=200,
        random_state=42 + fold
    )
    xgb_model.fit(
        X_xgb.iloc[tr_idx], y_tr,
        eval_set=[(X_xgb.iloc[va_idx], y_va)],
        verbose=False
    )
    oof['xgb'][va_idx] = xgb_model.predict_proba(X_xgb.iloc[va_idx])[:, 1]
    test_preds['xgb'] += xgb_model.predict_proba(X_test_xgb)[:, 1] / 5
    auc_xgb = roc_auc_score(y_va, oof['xgb'][va_idx])
    fold_scores['xgb'].append(auc_xgb)
    print(f"  XGBoost  AUC: {auc_xgb:.5f}  (iters: {xgb_model.best_iteration})")
    
    # ----------------------------------------------------------
    # 3. CatBoost — MEDIUM DEPTH, STRONG L2
    # ----------------------------------------------------------
    cat_model = CatBoostClassifier(
        iterations=5000,
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=5,
        eval_metric='AUC',
        random_seed=42 + fold,
        early_stopping_rounds=200,
        cat_features=cat_cols,
        verbose=False
    )
    cat_model.fit(
        X_cat.iloc[tr_idx], y_tr,
        eval_set=(X_cat.iloc[va_idx], y_va)
    )
    oof['cat'][va_idx] = cat_model.predict_proba(X_cat.iloc[va_idx])[:, 1]
    test_preds['cat'] += cat_model.predict_proba(X_test_cat)[:, 1] / 5
    auc_cat = roc_auc_score(y_va, oof['cat'][va_idx])
    fold_scores['cat'].append(auc_cat)
    print(f"  CatBoost AUC: {auc_cat:.5f}")
    
    # ----------------------------------------------------------
    # 4. RealMLP — NEURAL NETWORK DIVERSITY
    # ----------------------------------------------------------
    if HAS_REALMLP:
        try:
            mlp_model = RealMLP_TD_Classifier(
                n_epochs=256,
                batch_size=256,
                random_state=42 + fold
            )
            mlp_model.fit(
                X_mlp.iloc[tr_idx], y_tr
            )
            oof['mlp'][va_idx] = mlp_model.predict_proba(X_mlp.iloc[va_idx])[:, 1]
            test_preds['mlp'] += mlp_model.predict_proba(X_test_mlp)[:, 1] / 5
            auc_mlp = roc_auc_score(y_va, oof['mlp'][va_idx])
            fold_scores['mlp'].append(auc_mlp)
            print(f"  RealMLP  AUC: {auc_mlp:.5f}")
        except Exception as e:
            print(f"  RealMLP failed: {e}")
            # Fall back to zero — won't contribute to blend
            fold_scores['mlp'].append(0.0)
    
    elapsed = time.time() - t0
    print(f"  Fold time: {elapsed:.0f}s")

print("\n" + "=" * 65)
print("CROSS-VALIDATION COMPLETE")
print("=" * 65)

TRAINING 5-FOLD CROSS-VALIDATION

FOLD 1 / 5


In [ ]:
print("=" * 65)
print("INDIVIDUAL MODEL OOF PERFORMANCE")
print("=" * 65)

active_models = {}
for name in oof:
    total_auc = roc_auc_score(y, oof[name])
    mean_fold = np.mean(fold_scores[name])
    std_fold = np.std(fold_scores[name])
    
    if total_auc > 0.55:  # skip models that failed
        active_models[name] = oof[name]
        print(f"  {name.upper():8s} OOF AUC: {total_auc:.5f}  "
              f"(mean: {mean_fold:.5f} ± {std_fold:.5f})")
    else:
        print(f"  {name.upper():8s} SKIPPED (failed or not available)")

# Prediction correlation
print("\n--- Spearman Rank Correlations ---")
corr_df = pd.DataFrame({n: rankdata(active_models[n]) for n in active_models})
print(corr_df.corr(method='spearman').round(4))

In [ ]:
# 1. Hill climbing on probability predictions
print("=" * 65)
print("ENSEMBLE BLENDING")
print("=" * 65)

hc_weights, hc_auc = hill_climb_blend(active_models, y, n_rounds=300)
print(f"\nHill Climbing Weights: {hc_weights}")
print(f"Hill Climbing OOF AUC: {hc_auc:.5f}")

# 2. Hill climbing on RANK predictions (often better for AUC)
rank_oofs = {n: rankdata(active_models[n]) / len(y) for n in active_models}
hc_rank_weights, hc_rank_auc = hill_climb_blend(rank_oofs, y, n_rounds=300)
print(f"\nRank Hill Climbing Weights: {hc_rank_weights}")
print(f"Rank Hill Climbing OOF AUC: {hc_rank_auc:.5f}")

# 3. Simple rank average (baseline comparison)
simple_rank = np.mean([rankdata(active_models[n]) / len(y) for n in active_models], axis=0)
simple_rank_auc = roc_auc_score(y, simple_rank)
print(f"\nSimple Rank Average OOF AUC: {simple_rank_auc:.5f}")

# Pick the best strategy
results = {
    'hill_climb_prob': (hc_weights, hc_auc, 'prob'),
    'hill_climb_rank': (hc_rank_weights, hc_rank_auc, 'rank'),
    'simple_rank_avg': (None, simple_rank_auc, 'simple_rank'),
}
best_name = max(results, key=lambda k: results[k][1])
best_weights, best_auc, best_mode = results[best_name]

print(f"\n>>> Best Strategy: {best_name} (AUC: {best_auc:.5f})")

In [ ]:
# Build test predictions using the best blending strategy
if best_mode == 'prob':
    final_pred = sum(best_weights[n] * test_preds[n] for n in best_weights)
elif best_mode == 'rank':
    test_ranks = {n: rankdata(test_preds[n]) / len(test) for n in best_weights}
    final_pred = sum(best_weights[n] * test_ranks[n] for n in best_weights)
else:  # simple_rank
    final_pred = np.mean([rankdata(test_preds[n]) / len(test) for n in active_models], axis=0)

# Save submission
sub_path = '/kaggle/working/submission.csv' if os.path.exists('/kaggle/working') else 'submission.csv'
submission = pd.DataFrame({'id': test['id'], 'addicted_label': final_pred})
submission.to_csv(sub_path, index=False)

print(f"Submission saved to: {sub_path}")
print(f"Rows: {len(submission):,}")
print(f"Prediction range: [{final_pred.min():.4f}, {final_pred.max():.4f}]")
print(f"\nBest OOF ROC-AUC: {best_auc:.5f}")
print(f"Strategy: {best_name}")
if best_weights:
    for name, weight in sorted(best_weights.items(), key=lambda x: -x[1]):
        print(f"  {name}: {weight:.3f}")
print(f"\n{'='*65}")
print("DONE — Good luck on the leaderboard!")
print(f"{'='*65}")